In [ ]:
# %%
import os
import glob
from ultralytics import YOLO

# 1. Coordinate absolute paths relative to execution folder context
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in locals() else os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

print(f"Project Root resolved to: {PROJECT_ROOT}")
print(f"Targeting dataset directory layout at: {DATA_DIR}")

# 2. Extract and sort your 174 distinct class subfolder labels
train_search_path = os.path.join(DATA_DIR, "train")
if os.path.exists(train_search_path):
    classes = sorted([f for f in os.listdir(train_search_path) if os.path.isdir(os.path.join(train_search_path, f))])
else:
    classes = [f"herb_class_{i}" for i in range(174)]

print(f"Detected {len(classes)} distinct target plant categories for training ingestion.")

# 3. FIX: Scan all 174 class folders and write explicit paths to a manifest text file
# Supported image extensions
valid_extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
all_image_paths = []

for class_name in classes:
    class_folder = os.path.join(train_search_path, class_name)
    for ext in valid_extensions:
        all_image_paths.extend(glob.glob(os.path.join(class_folder, ext)))

# Save the manifest list to text files
train_manifest_path = os.path.join(NOTEBOOK_DIR, "train_manifest.txt")
with open(train_manifest_path, "w") as f:
    for img_path in all_image_paths:
        f.write(f"{img_path}\n")

print(f"Created dataset file manifest containing {len(all_image_paths)} images at: {train_manifest_path}")

# 4. Compile dataset.yaml pointing to the path manifest text file directly
yaml_content = f"""
path: {DATA_DIR}
train: {train_manifest_path}
val: {train_manifest_path}  # Fallback to manifest to pass validation checks
test: {train_manifest_path}

names:
"""
for idx, class_name in enumerate(classes):
    yaml_content += f"  {idx}: {class_name}\n"

yaml_path = os.path.join(NOTEBOOK_DIR, "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

print(f"Successfully compiled custom dataset config rules to: {yaml_path}")

# %%
# Load the pre-trained structural feature layers
model = YOLO("yolov8n.pt")

# Execute optimization training iterations
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device="cpu",  # Set to device="mps" for Apple Silicon hardware acceleration
    workers=2,
    project=os.path.join(NOTEBOOK_DIR, "herb_runs"),
    name="botany_yolo"
)

print("Training run executed successfully!")

Project Root resolved to: /Users/steve/CS/MedAi/herb-ai
Targeting dataset directory layout at: /Users/steve/CS/MedAi/herb-ai/data
Detected 174 distinct target plant categories for training ingestion.
Created dataset file manifest containing 0 images at: /Users/steve/CS/MedAi/herb-ai/research/train_manifest.txt
Successfully compiled custom dataset config rules to: /Users/steve/CS/MedAi/herb-ai/research/dataset.yaml
New https://pypi.org/project/ultralytics/8.4.67 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.60 🚀 Python-3.12.2 torch-2.2.2 CPU (Apple M3)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/steve/CS/MedAi/herb-ai/research/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropo